# Phase 6: Chronological Gaussian-Process Training Design

This notebook defines the weather-only training and validation design
used for Gaussian-process residual modelling.

The first 365 dates form the initial training period. The remaining 365
dates are divided into four consecutive validation blocks containing
91, 91, 91 and 92 dates. Training expands after each validation block.

The response variable is

\[
R_{d,r}
=
T_d^{\mathrm{HKO}}
-
\widehat T_{d,r}^{\mathrm{det}}.
\]

A separate Gaussian process will later be fitted for each decision rule.

No Gaussian process is fitted in this phase.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()

for candidate in (ROOT, *ROOT.parents):
    if (
        candidate
        / "config/v2/"
        "gp_training_design_spec.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Repository root not found."
    )

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/v2/"
            "build_gp_training_design.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)

    raise RuntimeError(
        "Phase 6 construction failed."
    )


PHASE 6 GP TRAINING DESIGN
Status: TWO_YEAR_GP_TRAINING_DESIGN_CERTIFIED
Weather-only dates: 730
Initial training dates: 365
Validation dates: 365
Validation blocks: [91, 91, 91, 92]
Decision rules: 4
Raw design rows: 2920
Fold matrix rows: 9484
Scaling parameter rows: 64
GP target: residual_c
GP features: calendar_time_years, seasonal_sin, seasonal_cos, forecast_daily_max_c
Separate model for each decision rule: True
Each validation date used once: True
Training precedes validation: True
Market prices accessed: False
Model fitted or selected: False
PHASE 6 CORE CONSTRUCTION: PASSED



In [2]:
import json
import pandas as pd

manifest = json.loads(
    (
        ROOT
        / "data/manifests/v2/"
        "06_gp_training_design_manifest.json"
    ).read_text(encoding="utf-8")
)

folds = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "06_gp_fold_summary.csv"
)

scaling = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "06_gp_fold_scaling_parameters.csv"
)

print("Status:", manifest["status"])
print()
print("Fold design:")
print(folds.to_string(index=False))
print()
print("First scaling rows:")
print(scaling.head(12).to_string(index=False))

Status: TWO_YEAR_GP_TRAINING_DESIGN_CERTIFIED

Fold design:
fold_id  fold_number training_start training_end validation_start validation_end  training_dates  validation_dates  training_rows  validation_rows
fold_01            1     2024-03-16   2025-03-15       2025-03-16     2025-06-14             365                91           1460              364
fold_02            2     2024-03-16   2025-06-14       2025-06-15     2025-09-13             456                91           1824              364
fold_03            3     2024-03-16   2025-09-13       2025-09-14     2025-12-13             547                91           2188              364
fold_04            4     2024-03-16   2025-12-13       2025-12-14     2026-03-15             638                92           2552              368

First scaling rows:
fold_id  fold_number decision_rule  decision_rule_order              feature  training_mean  training_standard_deviation  training_minimum  training_maximum  training_rows
fold_01     

In [3]:
import numpy as np
import pandas as pd

matrix = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "06_gp_fold_matrix_panel.csv"
)

assignments = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "06_gp_fold_date_assignments.csv"
)

assert len(matrix) == 9484

validation = assignments.loc[
    assignments["sample_role"].eq(
        "validation"
    )
]

assert len(validation) == 365
assert validation.groupby(
    "target_date"
).size().eq(1).all()

for (
    fold_id,
    decision_rule,
), group in matrix.loc[
    matrix["sample_role"].eq(
        "training"
    )
].groupby(
    ["fold_id", "decision_rule"],
    sort=False,
):
    for column in [
        "calendar_time_years_z",
        "seasonal_sin_z",
        "seasonal_cos_z",
        "forecast_daily_max_c_z",
    ]:
        assert abs(
            float(group[column].mean())
        ) <= 1e-10

        assert abs(
            float(group[column].std(ddof=0))
            - 1.0
        ) <= 1e-10

assert manifest["model_fitted"] is False
assert manifest["model_selected"] is False
assert manifest["market_prices_accessed"] is False

print("PHASE 6 NOTEBOOK VERIFICATION: PASSED")

PHASE 6 NOTEBOOK VERIFICATION: PASSED
